# Notebook 3 — Stochastic methods: paying less per step

**Day 3.** Read this after Lecture 3, with your `SGD` and `Adam` written.

Notebook 2 spent a full pass over the data to take one step. On $n = 10^6$ samples that
is unaffordable. The stochastic idea is to estimate the gradient from a handful of
samples, accept that the estimate is wrong, and take many more steps for the same price.

This notebook is about what you buy and what you pay:

1. the batch gradient is **unbiased**, and its variance falls like $1/b$ — measured;
2. per *epoch*, neither $b = 1$ nor $b = n$ wins;
3. a constant step size does not converge — it reaches a **noise floor**;
4. the floor's law depends on how you sample, and your `SGD` beats the textbook;
5. on an ill-conditioned problem Adam finishes and SGD does not — and we measure why.

Every number quoted in a caption is printed by the cell above it. Run them in order.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The optlab root is the nearest ancestor holding pyproject.toml, so this works whether
# Jupyter was started in notebooks/ or in the repository root.
HERE = Path.cwd()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "pyproject.toml").exists()), HERE.parent)

try:
    import optlab
except ModuleNotFoundError:
    sys.path.insert(0, str(ROOT / "src"))
    import optlab

sys.path.insert(0, str(ROOT))  # for datasets/

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
                     "axes.titlesize": 10, "figure.dpi": 110})
rng = np.random.default_rng(20250921)

print("optlab root  :", ROOT)
print("optlab loaded:", Path(optlab.__file__).parent)

In [ ]:
def status(name, thunk):
    """Report whether one piece of the package is implemented, without a traceback."""
    try:
        thunk()
    except NotImplementedError:
        return f"  MISSING   {name}"
    except Exception as err:                      # noqa: BLE001 - we want to see anything
        return f"  BROKEN    {name}   ({type(err).__name__}: {err})"
    return f"  ok        {name}"

In [ ]:
from optlab.losses import LogisticNLL, SquaredError
from optlab.observers import History
from optlab.optimizers import Adam, SGD, gradient_descent
from optlab.problems import GLMLoss, logistic_regression

_Xc, _yc = np.eye(2), np.array([1.0, 0.0])
_glm = GLMLoss(_Xc, _yc, SquaredError())
_w = np.zeros(2)

print("Day 3 readiness")
print(status("GLMLoss.n_samples", lambda: _glm.n_samples))
print(status("GLMLoss.batch_gradient", lambda: _glm.batch_gradient(_w, np.array([0]))))
print(status("SGD", lambda: SGD(batch_size=1, lr=0.1, n_epochs=1,
                                rng=np.random.default_rng(0)).minimize(_glm, _w)))
print(status("Adam", lambda: Adam(batch_size=1, lr=0.1, n_epochs=1,
                                  rng=np.random.default_rng(0)).minimize(_glm, _w)))
print()
print("Still needed from earlier days")
print(status("LogisticNLL", lambda: LogisticNLL().value(0.5, 1.0)))
print(status("GLMLoss.gradient", lambda: _glm.gradient(_w)))
print(status("gradient_descent (for the reference minimum)",
             lambda: gradient_descent(max_iter=3).minimize(_glm, _w)))

`MISSING` means the stub still raises `NotImplementedError`; go and write it. `BROKEN`
means your code ran and failed — the message tells you where to look.

## 1. The batch gradient is an unbiased estimate with variance $\propto 1/b$

The full gradient of an empirical risk is an average over the data:

$$\nabla f(w) \;=\; \frac{1}{n}\sum_{i=1}^{n} \nabla \ell_i(w).$$

Pick $b$ indices $\mathcal{B}$ *without replacement* and average over those instead:

$$g_{\mathcal{B}}(w) \;=\; \frac{1}{b}\sum_{i \in \mathcal{B}} \nabla \ell_i(w).$$

Two claims, both testable. First, $\mathbb{E}[g_{\mathcal{B}}] = \nabla f$ — the estimate
is **unbiased**, it is not systematically wrong in any direction. Second,

$$\mathbb{E}\bigl\|g_{\mathcal{B}} - \nabla f\bigr\|^2 \;=\; \frac{n-b}{(n-1)\,b}\,\sigma^2 ,
\qquad
\sigma^2 = \frac{1}{n}\sum_i \bigl\|\nabla \ell_i - \nabla f\bigr\|^2 .$$

The $\sigma^2$ is a property of the *data at this $w$*; you cannot change it. The
fraction is the only part you control. Note what it does at $b = n$: it is exactly zero,
because the "sample" is then the whole population.

We measure both by brute force: draw 3000 batches at $w = 0$ and compare.

In [ ]:
n, d = 400, 5

# A generator of its own, so re-running this cell gives you the same data every time.
data_rng = np.random.default_rng(20250921)
X = data_rng.normal(size=(n, d))
w_true = data_rng.normal(size=d)
y = (data_rng.random(n) < 1.0 / (1.0 + np.exp(-(X @ w_true)))).astype(float)

loss = logistic_regression(X, y)
w0 = np.zeros(d)

# A high-accuracy reference minimum, so "gap to the optimum" means something below.
# Day 2's gradient descent, run to a tight tolerance -- this is the recipe Labwork 3
# gives you in Plan B. `newton` would get there in a handful of iterations instead of
# 282, but you do not write it until day 4.
f_star = gradient_descent(tol=1e-9, max_iter=5000).minimize(loss, w0).value
print(f"logistic regression, n = {n}, d = {d},  f* = {f_star:.6f}")

g_full = loss.gradient(w0)
sampler = np.random.default_rng(1)
sizes = [1, 2, 4, 8, 16, 32, 64, 128, n]
measured = []

print(f"\n{'b':>5} {'||bias||':>11} {'measured var':>14} {'(n-b)/((n-1)b)':>16} {'ratio':>9}")
for bs in sizes:
    draws = np.array([loss.batch_gradient(w0, sampler.choice(n, bs, replace=False))
                      for _ in range(3000)])
    bias = np.linalg.norm(draws.mean(axis=0) - g_full)
    var = float(np.mean(np.sum((draws - g_full) ** 2, axis=1)))
    pred = (n - bs) / ((n - 1) * bs)
    measured.append(var)
    ratio = f"{var / pred:9.4f}" if pred else f"{'--':>9}"
    print(f"{bs:5d} {bias:11.2e} {var:14.4e} {pred:16.4e} {ratio}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

inner = sizes[:-1]
ax[0].loglog(inner, measured[:-1], "o-", label=r"measured $E\|g_B-\nabla f\|^2$")
sigma2 = measured[0] / ((n - 1) / (n - 1))            # the b=1 value pins the constant
law = [sigma2 * (n - bs) / ((n - 1) * bs) for bs in inner]
ax[0].loglog(inner, law, "k--", label=r"$\sigma^2\,(n-b)/((n-1)b)$, $\sigma^2$ from $b=1$")
ax[0].set_xlabel("batch size $b$")
ax[0].set_ylabel("variance")
ax[0].set_title("Variance of the batch gradient")
ax[0].legend(fontsize=8)

ax[1].semilogx(inner, [measured[i] / law[i] for i in range(len(inner))], "o-")
ax[1].axhline(1.0, color="k", ls="--", lw=1)
ax[1].set_xlabel("batch size $b$")
ax[1].set_ylabel("measured / law")
ax[1].set_ylim(0.9, 1.1)
ax[1].set_title("Ratio to the law (a flat line means the law holds)")

fig.suptitle(f"Figure 1 — {len(inner)} batch sizes, 3000 draws each, at $w=0$")
fig.tight_layout()
plt.show()

**Figure 1.** The left panel is a straight line of slope $-1$ over two decades: doubling
the batch halves the variance.

The right panel is the honest test. The law fixes the *shape* but not the constant
$\sigma^2$, so we read $\sigma^2$ off the $b = 1$ column and plot measured/predicted. The
leftmost point is therefore $1$ by construction and proves nothing; the claim is about
the other seven, which stay inside $\pm 1.5\%$ of it across two decades of $b$. A flat
line is the law holding. A line that drifted would mean the $1/b$ shape is wrong.

The `||bias||` column is small and shrinking like $1/\sqrt{3000}$ — that is Monte-Carlo
noise in our own measurement, not bias in the estimator. The last row settles it: at
$b = n$ there is one possible batch, the bias is $10^{-14}$ and the variance $10^{-32}$.
Both are zero up to floating point.

> **The trade you are making.** Halving the variance costs twice the work per step. But
> the variance enters the *error* under a square root, so four times the work buys you
> one bit. That is a terrible exchange rate, and it is why small batches win — see §2.

## 2. Count epochs, not iterations

Comparing $b = 1$ and $b = 128$ by iteration count is meaningless — the $b=1$ step is
128 times cheaper. The fair unit is the **epoch**: one pass over the data, i.e. the same
number of per-sample gradient evaluations whatever $b$ is. That is why your `SGD` takes
`n_epochs` and not `max_iter`.

First a sanity check that ties this notebook to the last one. At $b = n$ every batch is
the whole dataset, so SGD *is* gradient descent with a fixed step. Not approximately —
exactly, to the last bit.

In [ ]:
n_ep = 25
lr = 0.3

hist = History()
res = SGD(batch_size=n, lr=lr, n_epochs=n_ep,
          rng=np.random.default_rng(0), observers=[hist]).minimize(loss, w0)

w_manual = w0.copy()
for _ in range(n_ep):
    w_manual = w_manual - lr * loss.gradient(w_manual)

print("SGD with b = n :", res.x)
print("hand-written GD:", w_manual)
print(f"max |difference| = {np.max(np.abs(res.x - w_manual)):.3e}")

If that difference is not at the $10^{-16}$ level, your `SGD` is doing something extra —
a different step-size convention, a stray decay, a partial last batch handled wrongly.
Fix it here, before the noise makes it invisible.

Now the same budget of 40 epochs, spent at five different batch sizes.

In [ ]:
batch_sizes = [1, 8, 32, 128, n]
runs = {}

print(f"{'b':>5} {'steps/epoch':>12} {'steps total':>12} {'final gap':>13}")
for bs in batch_sizes:
    h = History()
    SGD(batch_size=bs, lr=0.3, n_epochs=40,
        rng=np.random.default_rng(3), observers=[h]).minimize(loss, w0)
    runs[bs] = h
    per_epoch = int(np.ceil(n / bs))
    print(f"{bs:5d} {per_epoch:12d} {per_epoch * 40:12d} {h.values[-1] - f_star:13.4e}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for bs in batch_sizes:
    gaps = np.maximum(np.array(runs[bs].values) - f_star, 1e-16)
    ax[0].semilogy(np.arange(1, len(gaps) + 1), gaps,
                   label=f"b = {bs}" + (" (= full GD)" if bs == n else ""))
ax[0].set_xlabel("epoch (one pass over the data)")
ax[0].set_ylabel(r"$f(w) - f^*$")
ax[0].set_title("Same budget, five batch sizes")
ax[0].legend(fontsize=8)

finals = [runs[bs].values[-1] - f_star for bs in batch_sizes]
ax[1].loglog(batch_sizes, finals, "o-")
for bs, fv in zip(batch_sizes, finals):
    ax[1].annotate(f"b={bs}", (bs, fv), textcoords="offset points",
                   xytext=(0, 8), ha="center", fontsize=8)
ax[1].set_xlabel("batch size $b$")
ax[1].set_ylabel(r"$f - f^*$ after 40 epochs")
ax[1].set_title("The cost of both extremes")

fig.suptitle("Figure 2 — epochs are the fair unit (lr = 0.3 throughout)")
fig.tight_layout()
plt.show()

**Figure 2.** The right panel is a U, and both walls are instructive.

At $b = n$ (400 samples per step, 40 steps total) the gap is $1.08\times10^{-2}$: the
steps are exact but there are only forty of them. At $b = 1$ (16 000 steps) the gap is
$5.09\times10^{-2}$ — *worse*, because each step uses a gradient estimated from a single
sample and the iterate rattles around the minimum instead of sitting in it.

The minimum is at $b = 32$, gap $2.42\times10^{-4}$ — 45 times better than $b = n$ and
210 times better than $b = 1$, for identical cost. This is the whole practical content of
mini-batching. The best $b$ is problem-dependent and hardware-dependent (on a GPU,
$b = 32$ and $b = 128$ often cost the *same* wall-clock time, which shifts the optimum
right), but the shape of the curve is always this U.

Now look at the left panel again, because it explains *why* the left wall of the U goes
up. The $b = 1$ curve flattens within a few epochs and then just rattles; $b = 8$ and
$b = 32$ flatten around epoch 15; $b = 128$ and $b = n$ are still descending at epoch 40
and have simply run out of budget. So the two walls of the U have entirely different
causes — the right one is *too few steps*, the left one is a floor that more steps cannot
get below. That floor is the subject of the next section.

## 3. The noise floor, and why your `SGD` beats the textbook

Gradient descent with a fixed step converges: notebook 2 showed the error shrinking by a
constant factor forever. Stochastic gradient descent with a fixed step **does not**. It
falls quickly, then stops at a level set by the step size — the *noise floor*.

The reason is a balance. The descent term pulls the error down in proportion to $\alpha$;
the noise injected by the sampling pushes it up in proportion to $\alpha^2 \sigma^2$.
They meet at a nonzero error. The standard textbook statement is

$$\lim_{k\to\infty}\ \mathbb{E}\,[\,f(w_k) - f^*\,] \;=\; O(\alpha) .$$

Let us test that. Four step sizes, long enough to plateau, and we average the gap over
the last quarter of the run so that we are measuring the floor and not one lucky iterate.

In [ ]:
def floor_of(lr, mode, b=8, sweeps=800, seed=7):
    """Average gap over the last quarter of a long constant-step run.

    mode="reshuffle" is what your SGD does: one permutation per epoch, so every
    sample is used exactly once. mode="iid" draws each batch independently, which
    is the assumption the O(alpha) analysis is built on.
    """
    gen = np.random.default_rng(seed)
    w = w0.copy()
    trace = []
    for _ in range(sweeps):
        if mode == "reshuffle":
            order = gen.permutation(n)
            batches = [order[i:i + b] for i in range(0, n, b)]
        else:
            batches = [gen.integers(0, n, size=b) for _ in range(n // b)]
        for idx in batches:
            w = w - lr * loss.batch_gradient(w, idx)
        trace.append(loss.value(w) - f_star)
    trace = np.array(trace)
    return float(trace[3 * len(trace) // 4:].mean()), trace


lrs = [0.4, 0.2, 0.1, 0.05]
floors, traces = {}, {}
for mode in ("reshuffle", "iid"):
    for lr in lrs:
        floors[mode, lr], traces[mode, lr] = floor_of(lr, mode)

print(f"{'lr':>6} {'reshuffle':>13} {'iid':>13}")
for lr in lrs:
    print(f"{lr:6.2f} {floors['reshuffle', lr]:13.4e} {floors['iid', lr]:13.4e}")

slopes = {m: np.polyfit(np.log(lrs), np.log([floors[m, lr] for lr in lrs]), 1)[0]
          for m in ("reshuffle", "iid")}
print(f"\nlog-log slope   reshuffle {slopes['reshuffle']:.2f}   iid {slopes['iid']:.2f}")
print("(the textbook O(alpha) prediction is a slope of 1)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

for lr in lrs:
    ax[0].semilogy(traces["reshuffle", lr], lw=0.8, label=f"lr = {lr}")
    ax[0].axhline(floors["reshuffle", lr], color="k", ls=":", lw=0.8)
ax[0].set_xlabel("sweep over the data")
ax[0].set_ylabel(r"$f(w) - f^*$")
ax[0].set_title("Four step sizes, four floors\n(dotted: the measured floor)")
ax[0].legend(fontsize=8)

for mode, marker in (("reshuffle", "o-"), ("iid", "s-")):
    vals = [floors[mode, lr] for lr in lrs]
    ax[1].loglog(lrs, vals, marker, label=f"{mode}   slope {slopes[mode]:.2f}")
# Anchor the reference at the reshuffling run's largest step, so the gap between the
# blue curve and the dashed line *is* the improvement over the textbook rate.
ref = np.array(lrs) * floors["reshuffle", lrs[0]] / lrs[0]
ax[1].loglog(lrs, ref, "k--", lw=1,
             label=r"slope 1 ($O(\alpha)$) from the same start")
ax[1].set_xlabel(r"step size $\alpha$")
ax[1].set_xticks(lrs)
ax[1].set_xticklabels([str(v) for v in lrs])
ax[1].minorticks_off()
ax[1].set_ylabel("noise floor")
ax[1].set_title("How the floor depends on the step size")
ax[1].legend(fontsize=8)

h_const, h_decay = History(), History()
SGD(batch_size=8, lr=0.4, n_epochs=400,
    rng=np.random.default_rng(7), observers=[h_const]).minimize(loss, w0)
SGD(batch_size=8, lr=0.4, n_epochs=400, lr_decay=0.2,
    rng=np.random.default_rng(7), observers=[h_decay]).minimize(loss, w0)
for h, lab in ((h_const, "constant lr = 0.4"), (h_decay, "lr = 0.4, lr_decay = 0.2")):
    ax[2].semilogy(np.maximum(np.array(h.values) - f_star, 1e-16), lw=0.9, label=lab)
ax[2].set_xlabel("epoch")
ax[2].set_ylabel(r"$f(w) - f^*$")
ax[2].set_title("Decaying the step removes the floor")
ax[2].legend(fontsize=8)

fig.suptitle("Figure 3 — the noise floor (b = 8)")
fig.tight_layout()
plt.show()

print(f"after 400 epochs:  constant {h_const.values[-1] - f_star:.4e}"
      f"   decaying {h_decay.values[-1] - f_star:.4e}")
print(f"the decaying run's last step was {0.4 / (1 + 0.2 * 399):.5f}, "
      f"down from {0.4} -- a factor of {1 + 0.2 * 399:.0f}")

**Figure 3, left.** Four runs, four plateaus. Nothing is converging; each run settles at
a level and stays there. Running longer buys nothing — only a smaller step does.

**Figure 3, middle — the interesting one.** The i.i.d. curve has measured slope 1.05,
which is the textbook $O(\alpha)$. The reshuffling curve is far steeper: 2.71 here. Your
`SGD` reshuffles, so **your implementation has a substantially better floor than the
theory predicts**, and at $\alpha = 0.05$ it is 66 times lower than the i.i.d. version.

Why: over one reshuffled sweep every sample is used exactly once, so the sampling errors
of the individual batches must sum to zero. They cancel. Under i.i.d. sampling they do
not — a sample can be drawn three times in a sweep and its neighbour not at all. This is
a real and well-studied effect (the literature calls it *random reshuffling*), and it is
why essentially every implementation in every framework shuffles rather than sampling.

Do **not** take 2.71 as a law. Rerun this with a different seed, a different $n$ or a
different problem and you will get a different exponent — on a larger version of this
problem it comes out near 1.8. What is robust is the *comparison*: reshuffling is
strictly better than i.i.d., and the textbook rate describes the version nobody uses.

> This is worth pausing on. The theory was not wrong; it answered a question about a
> different algorithm than the one in your file. When a measurement disagrees with a
> bound, the first suspect is always the bound's hypotheses.

**Figure 3, right.** The other way to remove the floor: shrink $\alpha$ as you go. Your
`lr_decay` implements $\alpha_k = \alpha_0/(1 + \gamma k)$ with $k$ the **epoch** index,
which satisfies the Robbins–Monro conditions $\sum \alpha_k = \infty$ (enough total
travel to reach the minimum) and $\sum \alpha_k^2 < \infty$ (little enough total noise to
settle). After 400 epochs the constant-step run sits at $6.3\times10^{-3}$; the decaying
one has reached $7.9\times10^{-9}$ — nearly six orders of magnitude, from one extra
argument.

Read the second printed line before choosing a $\gamma$ of your own. Over 400 epochs
$\gamma = 0.2$ divides the step by $1 + 0.2\times399 \approx 80$, which is what takes the
run from the $\alpha = 0.4$ floor down past the $\alpha = 0.05$ one in the middle panel.
A tenth of that, $\gamma = 0.02$, would divide it by only $9$, and the run would stall at
the floor belonging to $\alpha \approx 0.045$ — $5.6\times10^{-6}$ here, 700 times
worse. The schedule is asymptotically right for any positive $\gamma$; on a
finite budget $\gamma$ decides where you actually stop.

## 4. Where Adam earns its place

Adam is usually sold as "SGD with adaptive learning rates", which explains nothing. Here
is the concrete case it fixes.

Build a linear regression whose two features live on wildly different scales — one of
order $1$, one of order $10^{-3}$. This is not exotic; it is what happens when one column
is a count and another is a probability, and nobody standardised. Notebook 1 §4 showed
what that does to $\kappa$.

In [ ]:
n2 = 1500
scale_rng = np.random.default_rng(4242)
X2 = np.column_stack([scale_rng.normal(size=n2),
                      scale_rng.normal(scale=1e-3, size=n2)])
w_true2 = np.array([2.0, -1500.0])            # the small feature needs a huge weight
y2 = X2 @ w_true2 + scale_rng.normal(scale=0.1, size=n2)

problem = GLMLoss(X2, y2, SquaredError())
w0_2 = np.zeros(2)

H = X2.T @ X2 / n2
eigs = np.linalg.eigvalsh(H)
w_star2 = np.linalg.solve(H, X2.T @ y2 / n2)
f_star2 = problem.value(w_star2)

print(f"eigenvalues of the Hessian : {eigs}")
print(f"condition number kappa     : {eigs[-1] / eigs[0]:.4e}")
print(f"optimum w*                 : {w_star2}")
print(f"f(w*)                      : {f_star2:.6e}")

In [ ]:
settings = [("SGD", SGD, [0.5, 1.0, 1.8]), ("Adam", Adam, [0.5, 2.0, 10.0])]
adaptive = {}

print(f"{'optimizer':>10} {'lr':>6} {'gap after 60 epochs':>22} {'best gap seen':>16}")
for name, cls, lr_list in settings:
    for lr in lr_list:
        h = History()
        cls(batch_size=32, lr=lr, n_epochs=60,
            rng=np.random.default_rng(1), observers=[h]).minimize(problem, w0_2)
        adaptive[name, lr] = h
        gaps = np.array(h.values) - f_star2
        print(f"{name:>10} {lr:6.1f} {gaps[-1]:22.4e} {gaps.min():16.4e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for (name, lr), h in adaptive.items():
    style = "-" if name == "Adam" else "--"
    ax.semilogy(np.maximum(np.array(h.values) - f_star2, 1e-12), style,
                lw=1.2, label=f"{name}, lr = {lr}")
ax.set_xlabel("epoch")
ax.set_ylabel(r"$f(w) - f^*$")
ax.set_title(r"Figure 4 — $\kappa \approx 9.6\times 10^{5}$: SGD stalls, Adam does not"
             "\n(dashed: SGD, solid: Adam, b = 32)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

**Figure 4.** Three SGD runs spanning a factor of almost four in step size end up in the
same place: gaps of $1.162$, $1.198$ and $1.642$, against a *best gap seen* of $1.146$,
$1.146$ and $1.144$. They are not converging slowly, they have stopped. Raising the step
does not help, because the largest usable step is set by the *stiff* direction (notebook
2: $\alpha < 2/L$, and $L = 0.973$ here, so $\alpha = 1.8$ is already near the edge —
which is why that run ends *above* its own best), while all the missing progress is in
the *soft* direction. There is no single number that serves both.

Adam, with the same batches and the same budget, reaches $4.23\times10^{-4}$ at
`lr = 2.0` — 2700 times better than the best SGD run. And note the value: `lr = 2.0`
would make SGD explode instantly. Adam's `lr` is not a step size in the same units, which
is the most common source of confusion about it. §5 shows exactly what it is instead.

**Adam has a sweet spot too.** Read the `lr = 10.0` row: final gap $7.90\times10^{-2}$,
but best gap seen $1.18\times10^{-5}$ — it found a better point than any other run and
then bounced away from it. The brown curve in the figure shows this as a wide, ragged
band. Adam removes the *conditioning* problem, not the noise-floor problem of §3; too
large an `lr` still buys you a high floor. "Use Adam and stop tuning" is not a thing.

## 5. What Adam is actually doing

Adam keeps two running averages, of the gradient and of its square, and divides one by
the square root of the other:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \qquad
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2,$$
$$w_{t+1} = w_t - \alpha \,\frac{\hat m_t}{\sqrt{\hat v_t} + \varepsilon},
\qquad \hat m_t = \frac{m_t}{1-\beta_1^t}, \quad \hat v_t = \frac{v_t}{1-\beta_2^t}.$$

The division is per-coordinate, and that is the whole trick. Look at the very first step,
where $m_1 = (1-\beta_1) g_1$ and $v_1 = (1-\beta_2) g_1^2$, so after bias correction
$\hat m_1 = g_1$ and $\sqrt{\hat v_1} = |g_1|$ and the ratio is $\operatorname{sign}(g_1)$
exactly. Let us check that against the arithmetic.

In [ ]:
g1 = problem.gradient(w0_2)
beta1, beta2, eps = 0.9, 0.999, 1e-8

m1 = (1 - beta1) * g1
v1 = (1 - beta2) * g1 ** 2

print("gradient at w0          ", g1)
for lr_a in (2.0, 10.0):
    step_corrected = lr_a * (m1 / (1 - beta1)) / (np.sqrt(v1 / (1 - beta2)) + eps)
    print(f"  lr = {lr_a:<5} first step {step_corrected}"
          f"   lr*sign(g) {lr_a * np.sign(g1)}")

lr_a = 2.0
step_corrected = lr_a * (m1 / (1 - beta1)) / (np.sqrt(v1 / (1 - beta2)) + eps)
step_raw = lr_a * m1 / (np.sqrt(v1) + eps)
print()
print("at lr = 2.0, without bias correction ", step_raw)
print("ratio raw / corrected                ", step_raw / step_corrected)
print(f"predicted (1-b1)/sqrt(1-b2)           {(1 - beta1) / np.sqrt(1 - beta2):.6f}")

The gradient components differ by three orders of magnitude — $-1.879$ and
$+1.452\times10^{-3}$ — and Adam's first step is $\pm\,$`lr` in **both**. It has thrown
the magnitude away and kept only the sign. That is why it crosses 1500 units of $w_2$
while SGD is still taking $\alpha \cdot 10^{-3}$-sized nudges there — and also why `lr`
is measured in units of $w$, not in units of $w$ per gradient.

The bias-correction check is worth reading twice. Without it the first step is $3.162$
times *larger*, matching $(1-\beta_1)/\sqrt{1-\beta_2}$ exactly. The usual story — "the
averages start at zero, so early steps are too small" — is only half true: $m$ and $v$
are both biased towards zero, the step is their ratio, and which bias wins depends on
$\beta_1$ against $\beta_2$. The correction is what makes the first step independent of
both.

Now watch the two coordinates move.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
pairs = [("SGD", 1.0), ("Adam", 2.0), ("Adam", 10.0)]

for coord, axis, target in ((0, ax[0], w_star2[0]), (1, ax[1], w_star2[1])):
    for name, lr in pairs:
        path = np.array([e.x[coord] for e in adaptive[name, lr].events])
        axis.plot(np.arange(1, len(path) + 1), path, label=f"{name}, lr = {lr}")
    axis.axhline(target, color="k", ls="--", lw=1, label=f"$w^*$ = {target:.3f}")
    axis.set_xlabel("epoch")
    axis.set_title(f"$w_{coord + 1}$ (well-scaled feature)" if coord == 0
                   else f"$w_{coord + 1}$ (feature scaled by $10^{{-3}}$)")
    axis.legend(fontsize=8)

fig.suptitle("Figure 5 — the same 60 epochs, one coordinate at a time")
fig.tight_layout()
plt.show()

for name, lr in pairs:
    w1_path = np.array([e.x[0] for e in adaptive[name, lr].events])
    path = np.array([e.x[1] for e in adaptive[name, lr].events])
    hit = np.where(np.abs(path - w_star2[1]) < 0.01 * abs(w_star2[1]))[0]
    print(f"{name:5s} lr={lr:<5} w2 after 1 / 10 / 60 epochs: "
          f"{path[0]:10.3f} {path[9]:10.3f} {path[-1]:10.3f}"
          f"   epochs to reach 1% of w*: {hit[0] + 1 if len(hit) else 'never'}")
print()
for name, lr in pairs:
    w1_path = np.array([e.x[0] for e in adaptive[name, lr].events])
    print(f"{name:5s} lr={lr:<5} w1 stays in [{w1_path.min():.3f}, {w1_path.max():.3f}]"
          f"   (width {w1_path.max() - w1_path.min():.3f})")

**Figure 5.** All three methods keep $w_1$ near its optimum $1.994$ and none of them is
making progress there any more; the widths printed below the figure say SGD rattles over
a band of width $1.65$, wider than either Adam run. So the left panel is *not* evidence
that Adam pays for §4 with a worse $w_1$ — on this problem it does not.

The right panel is the whole story. SGD's $w_2$ goes $-0.07$, $-0.72$, $\dots$, $-4.32$
after sixty epochs, on its way to $-1503.6$: it has covered $0.3\%$ of the distance and
is slowing down, and the printout says it reaches 1% of the target *never*. Adam at
`lr = 10.0` is already at $-422$ after a single epoch and inside 1% by epoch 8; at
`lr = 2.0` it takes 38 epochs. Five times the step, about a fifth of the epochs — the
`lr` sets the crossing speed, just as the sign-step picture says it should.

That is also the trade-off from §4 in one picture: `lr = 10.0` crosses the distance five
times faster and then cannot stop precisely, while `lr = 2.0` arrives later and settles.

Tie this back to notebook 1 §4. There, the cure for $\kappa = 1.1\times 10^{6}$ was to
standardise the columns, and $\kappa$ fell to $1.87$. That remains the right fix — it is
cheaper, it is one line, and it leaves you with a problem that plain SGD solves. Adam is
what you reach for when you *cannot* fix the conditioning: when the scales differ per
layer of a network, or drift during training, or you simply do not know them. It buys
robustness to bad conditioning, and it pays for it with a noise floor that adaptive
methods are notoriously bad at lowering.

## Checkpoint

You should be able to answer these from cells you ran, not from memory.

1. Your colleague reports that going from $b = 32$ to $b = 256$ "made each epoch converge
   worse". Is that a bug? Which panel of Figure 2 settles it?
2. A run has plateaued at $f - f^* \approx 10^{-3}$ and has been there for 200 epochs.
   You have two knobs, `lr` and `lr_decay`. Which one, and what do you expect to see?
3. The textbook says the noise floor is $O(\alpha)$ and your measurement says
   $O(\alpha^{2.7})$. Without rerunning anything, what is the *one* line of `floor_of`
   that explains the discrepancy?
4. Someone sets Adam's `lr = 0.3` because "that worked for SGD". Using §5, what is Adam's
   first step in units of $w$, and roughly how many epochs would it then need to cross
   the 1500 units of $w_2$?
5. Turn the bias correction off in your own `Adam` and rerun §4. Does the 60-epoch gap
   get better or worse, and does the *first* step get bigger or smaller? (The second
   question you can answer before running it.)

**Before day 4.** Your `Quadratic.hessian` and `GLMLoss.hessian` are needed from the
first cell of notebook 4 onwards. `GLMLoss.hessian` is $X^\top D X / n$ with $D$
diagonal, and the entries of $D$ come from `IPointwiseLoss.second_derivative` — you wrote
that on day 1 and have not used it since.